# 03 Deskew and HOG

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_004_SVC_Normal_Deskew_SafePilot.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)
- [`EXP_005_SVC_Normal_HOG_SafePilot.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '58346708745207b057b4e626ced2597edf647194'}, '05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '7c8220ef85b8f24d5dcdaefc56d137c13b42692e'}}

## EXP_004_SVC_Normal_Deskew_SafePilot

원본: [05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py)  
Git blob: `58346708745207b057b4e626ced2597edf647194`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-004 safety pilot: CoM versus CoM followed by Deskew.

Run from repo root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py --train-size 16000

This is a reduced-data development pilot, NOT a full 76,800-row experiment.
Never reads official Test. Does not overwrite EXP-001/002/003 result folders.

CLI 인자 (정적 추출): `--train-size`, `--val-size`, `--batch-size`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-004 safety pilot: CoM versus CoM followed by Deskew.

Run from repo root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py
  python 05_model_optimize/SVC/Normal/scripts/EXP_004_SVC_Normal_Deskew_SafePilot.py --train-size 16000

This is a reduced-data development pilot, NOT a full 76,800-row experiment.
Never reads official Test. Does not overwrite EXP-001/002/003 result folders.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from common import fixed_split, pipeline

OUT_BASE = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'


def say(*items):
    print(*items, flush=True)


def select_stratified(x, y, n, seed=42):
    if n >= len(y):
        return x, y
    ix, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=seed)
    return x[ix], y[ix]


def batched_predict(model, x, name, batch=256):
    output = []
    start = time.perf_counter()
    for i in range(0, len(x), batch):
        p = model.predict(x[i:i + batch]).astype(np.int64)
        output.append(p)
        if (i // batch) % 8 == 0:
            say(f'  {name}: {min(i + batch, len(x))}/{len(x)} predicted')
    return np.concatenate(output), round(time.perf_counter() - start, 2)


def run(args):
    svc, scaling, source = baseline()
    if not scaling:
        raise RuntimeError('Original project SVC baseline must use scaling')
    say('SVC baseline:', source)
    say('SVC params:', svc.get_params())
    say('EXP-004 SAFE PILOT. CoM vs CoM->Deskew. Official Test not loaded.')
    test_deskew()
    t = time.perf_counter()
    xtr, ytr, xval, yval, sha = fixed_split()
    say(f'Official split loaded in {time.perf_counter() - t:.2f} s: {len(ytr)} / {len(yval)}')
    xtr, ytr = select_stratified(xtr, ytr, args.train_size)
    xval, yval = select_stratified(xval, yval, args.val_size)
    say(f'Pilot train={len(ytr)} val={len(yval)}; X dtype={xtr.dtype}')
    hm = {w: load_writer(w) for w in WRITERS}
    com = BatchedCoM()
    t = time.perf_counter()
    centered_train = com.transform(xtr)
    centered_val = com.transform(xval)
    centered_hm = {w: (com.transform(hm[w][0]), hm[w][1]) for w in WRITERS}
    com_seconds = round(time.perf_counter() - t, 2)
    say('CoM preprocessed in', com_seconds, 'sec')
    t = time.perf_counter()
    deskew_train, deskew_train_stats = deskew(centered_train)
    deskew_val, deskew_val_stats = deskew(centered_val)
    deskew_hm = {}
    for w in WRITERS:
        adjusted, _ = deskew(centered_hm[w][0])
        deskew_hm[w] = (adjusted, centered_hm[w][1])
    deskew_seconds = round(time.perf_counter() - t, 2)
    say('Deskew after CoM preprocessed in', deskew_seconds, 'sec')
    if not np.isfinite(deskew_train).all() or deskew_train.shape != centered_train.shape:
        raise RuntimeError('Deskew train invalid')
    if not np.isfinite(centered_train).all():
        raise RuntimeError('Nonfinite CoM transformed train')

    records = {}
    for label, train, val, hw in [
        ('com_subset', centered_train, centered_val, centered_hm),
        ('com_deskew_subset', deskew_train, deskew_val, deskew_hm),
    ]:
        clf = pipeline(clone(svc), scaling)
        say(f'[{label}] fit START')
        t = time.perf_counter()
        clf.fit(train, ytr)
        fit_seconds = round(time.perf_counter() - t, 2)
        say(f'[{label}] fit DONE {fit_seconds}s')
        pv, predict_seconds = batched_predict(clf, val, label + ' official', args.batch_size)
        official = report(yval, pv)
        say(f'[{label}] official={official["accuracy"]:.6f}, F1={official["macro_f1"]:.6f} predict={predict_seconds}s')
        writers = []
        truths, preds = [], []
        for w in WRITERS:
            x, y = hw[w]
            p, pred_sec = batched_predict(clf, x, label + f' writer {w}', args.batch_size)
            m = report(y, p)
            writers.append({'writer': w, **m, 'predict_seconds': pred_sec})
            truths.append(y)
            preds.append(p)
            say(f'[{label}] writer {w}: {m["accuracy"]:.6f}')
        handmade = report(np.concatenate(truths), np.concatenate(preds))
        say(f'[{label}] Hand-made={handmade["accuracy"]:.6f} F1={handmade["macro_f1"]:.6f}')
        records[label] = {'fit_seconds': fit_seconds, 'official_validation': official,
                          'official_predict_seconds': predict_seconds,
                          'handmade_development': handmade, 'writers': writers}

    out = OUT_BASE / f'EXP_004_SVC_Normal_Deskew_SAFE_PILOT_train{len(ytr)}'
    out.mkdir(parents=True, exist_ok=True)
    comparison = records['com_deskew_subset']['handmade_development']['accuracy'] - records['com_subset']['handmade_development']['accuracy']
    off_delta = records['com_deskew_subset']['official_validation']['accuracy'] - records['com_subset']['official_validation']['accuracy']
    summary = {
        'experiment': 'EXP-004 SVC CoM -> Deskew SAFE PILOT (subset only)',
        'train_size': len(ytr), 'validation_size': len(yval),
        'com_preprocessing_seconds': com_seconds, 'deskew_preprocessing_seconds': deskew_seconds,
        'deskew_train_diagnostics': deskew_train_stats,
        'deskew_validation_diagnostics': deskew_val_stats,
        'official_test_used': False,
        'model_source': source, 'hyperparameters': svc.get_params(),
        'official_dataset_sha256': sha, 'results': records,
        'handmade_delta_pp': 100 * comparison,
        'official_delta_pp': 100 * off_delta,
        'note': '8,000-row development pilot; do not directly compare to 76,800-row full baseline',
    }
    (out / 'metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    with (out / 'comparison.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['model', 'fit_seconds', 'official_accuracy', 'official_macro_f1', 'handmade_accuracy', 'handmade_macro_f1'])
        writer.writeheader()
        for name, r in records.items():
            writer.writerow({'model': name, 'fit_seconds': r['fit_seconds'],
                             'official_accuracy': r['official_validation']['accuracy'],
                             'official_macro_f1': r['official_validation']['macro_f1'],
                             'handmade_accuracy': r['handmade_development']['accuracy'],
                             'handmade_macro_f1': r['handmade_development']['macro_f1']})
    say('PILOT COMPARISON: CoM+Deskew minus CoM official:', f'{summary["official_delta_pp"]:+.4f}pp',
        'Hand-made:', f'{summary["handmade_delta_pp"]:+.4f}pp')
    say('Saved:', out)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train-size', type=int, default=8000, choices=(2000, 4000, 8000, 16000))
    parser.add_argument('--val-size', type=int, default=3200, choices=(800, 1600, 3200, 6400))
    parser.add_argument('--batch-size', type=int, default=256, choices=(64, 128, 256, 512))
    run(parser.parse_args())


## EXP_005_SVC_Normal_HOG_SafePilot

원본: [05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py)  
Git blob: `7c8220ef85b8f24d5dcdaefc56d137c13b42692e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

EXP-005 SVC Normal: CoM+Deskew with pixel, HOG and pixel:HOG (20:80).

Run from the Machine_Learning repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py

Safe development pilot: 8,000 official train / 3,200 official validation,
all 2,560 Hand-made for development, and no official Test data.
Reuses EXP-004 preprocessing and EXP-012 HOG implementation.

CLI 인자 (정적 추출): `--train-size`, `--val-size`, `--batch-size`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-005 SVC Normal: CoM+Deskew with pixel, HOG and pixel:HOG (20:80).

Run from the Machine_Learning repository root:
  python 05_model_optimize/SVC/Normal/scripts/EXP_005_SVC_Normal_HOG_SafePilot.py

Safe development pilot: 8,000 official train / 3,200 official validation,
all 2,560 Hand-made for development, and no official Test data.
Reuses EXP-004 preprocessing and EXP-012 HOG implementation.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP004_KNN_Normal_CoM import BatchedCoM
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, hybrid_features, test_hog
from common import fixed_split, pipeline

RESULTS = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'


def say(*parts):
    print(*parts, flush=True)


def select_stratified(x, y, n, seed=42):
    if n > len(y):
        raise ValueError(f'Requested {n} rows but only {len(y)} exist')
    if n == len(y):
        return x, y
    ix, _ = train_test_split(np.arange(len(y)), train_size=n,
                             stratify=y, random_state=seed)
    return x[ix], y[ix]


def predict_batches(model, x, name, batch_size=256):
    preds = []
    start = time.perf_counter()
    for pos in range(0, len(x), batch_size):
        end = min(pos + batch_size, len(x))
        preds.append(np.asarray(model.predict(x[pos:end]), dtype=np.int64))
        if pos == 0 or end == len(x) or (pos // batch_size + 1) % 8 == 0:
            say(f'  {name}: {end}/{len(x)} predicted')
    return np.concatenate(preds), round(time.perf_counter() - start, 2)


def run(args):
    svc, scaling, source = baseline()
    if not scaling:
        raise RuntimeError('Original SVC baseline unexpectedly has scaling=False')
    say('SVC baseline:', source)
    say('SVC parameters:', svc.get_params())
    say('EXP-005 SAFE PILOT: pixel vs HOG vs pixel:HOG 20:80; official Test NOT loaded')
    test_deskew()
    test_hog()

    t = time.perf_counter()
    xtr, ytr, xval, yval, sha = fixed_split()
    say(f'Official split loaded in {time.perf_counter()-t:.2f}s: {len(ytr)} / {len(yval)}')
    xtr, ytr = select_stratified(xtr, ytr, args.train_size)
    xval, yval = select_stratified(xval, yval, args.val_size)
    writers = {w: load_writer(w) for w in WRITERS}
    say(f'Pilot train={len(ytr)} val={len(yval)} handmade={sum(len(v[1]) for v in writers.values())}')

    t = time.perf_counter()
    com = BatchedCoM()
    centered_train = com.transform(xtr)
    centered_val = com.transform(xval)
    centered_writers = {w: (com.transform(writers[w][0]), writers[w][1]) for w in WRITERS}
    say(f'CoM preprocessing completed in {time.perf_counter()-t:.2f}s')
    t = time.perf_counter()
    pixel_train, _ = deskew(centered_train)
    pixel_val, _ = deskew(centered_val)
    pixel_writers = {w: (deskew(centered_writers[w][0])[0], centered_writers[w][1])
                     for w in WRITERS}
    say(f'Deskew preprocessing completed in {time.perf_counter()-t:.2f}s')

    t = time.perf_counter()
    hog_train = hog_features(pixel_train)
    hog_val = hog_features(pixel_val)
    hog_writers = {w: (hog_features(pixel_writers[w][0]), pixel_writers[w][1])
                   for w in WRITERS}
    say(f'HOG features extracted in {time.perf_counter()-t:.2f}s')
    t = time.perf_counter()
    mix_train = hybrid_features(pixel_train, hog_train, pixel_weight=0.20)
    mix_val = hybrid_features(pixel_val, hog_val, pixel_weight=0.20)
    mix_writers = {w: (hybrid_features(pixel_writers[w][0], hog_writers[w][0], pixel_weight=0.20),
                       pixel_writers[w][1]) for w in WRITERS}
    say(f'Pixel:HOG 20:80 features prepared in {time.perf_counter()-t:.2f}s')

    # The project pipeline applies 0..255 -> 0..1 scaling. This is only
    # appropriate for raw pixel features. HOG and hybrid are already normalized,
    # so we pass them directly to the same original SVC estimator.
    candidates = [
        ('pixel_com_deskew', pixel_train, pixel_val, pixel_writers, True),
        ('hog_com_deskew', hog_train, hog_val, hog_writers, False),
        ('pixel20_hog80_com_deskew', mix_train, mix_val, mix_writers, False),
    ]
    output_dir = RESULTS / f'EXP_005_SVC_Normal_HOG_SAFE_PILOT_train{len(ytr)}'
    output_dir.mkdir(parents=True, exist_ok=True)
    records = {}
    summary_rows = []

    for name, train, val, hw, use_scaler in candidates:
        model = pipeline(clone(svc), True) if use_scaler else clone(svc)
        assert train.shape[1] in (784, 324, 1108)
        assert np.isfinite(train).all() and np.isfinite(val).all()
        say(f'\n[{name}] fit START; rows={len(ytr)} dims={train.shape[1]} scaled_by_project_pipeline={use_scaler}')
        t = time.perf_counter()
        model.fit(train, ytr)
        fit_s = round(time.perf_counter() - t, 2)
        say(f'[{name}] fit DONE {fit_s}s')
        official_pred, official_pred_s = predict_batches(model, val, f'{name} official', args.batch_size)
        official_result = report(yval, official_pred)
        say(f'[{name}] Official Validation accuracy={official_result["accuracy"]:.6f} '
            f'F1={official_result["macro_f1"]:.6f}; predict={official_pred_s}s')
        by_writer, y_all, p_all = [], [], []
        for w in WRITERS:
            wx, wy = hw[w]
            wp, wp_s = predict_batches(model, wx, f'{name} writer {w}', args.batch_size)
            scores = report(wy, wp)
            by_writer.append({'writer': w, 'accuracy': scores['accuracy'],
                              'macro_f1': scores['macro_f1'], 'n': len(wy),
                              'predict_seconds': wp_s})
            y_all.append(wy)
            p_all.append(wp)
            say(f'[{name}] writer {w} accuracy={scores["accuracy"]:.6f}')
        hand_result = report(np.concatenate(y_all), np.concatenate(p_all))
        say(f'[{name}] Hand-made development accuracy={hand_result["accuracy"]:.6f} '
            f'F1={hand_result["macro_f1"]:.6f}')
        records[name] = {
            'feature_dimensions': int(train.shape[1]),
            'scaling': 'project pixel/255 pipeline' if use_scaler else 'pre-normalized features; no pixel/255 scaler',
            'fit_seconds': fit_s, 'official_predict_seconds': official_pred_s,
            'official_validation': official_result, 'handmade_development': hand_result,
            'writers': by_writer,
        }
        summary_rows.append({
            'model': name, 'features': int(train.shape[1]), 'train_size': len(ytr),
            'fit_seconds': fit_s, 'official_accuracy': official_result['accuracy'],
            'official_macro_f1': official_result['macro_f1'],
            'handmade_accuracy': hand_result['accuracy'],
            'handmade_macro_f1': hand_result['macro_f1'],
        })
        (output_dir / 'metrics_partial.json').write_text(
            json.dumps(records, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        del model

    control = records['pixel_com_deskew']
    # This pilot fixes the same stratified subsets as EXP-004 and should
    # reproduce the saved control accuracy for 8,000/3,200.
    if args.train_size == 8000 and args.val_size == 3200:
        expect = (0.959375, 0.76796875)
        found = (control['official_validation']['accuracy'],
                 control['handmade_development']['accuracy'])
        if not np.allclose(found, expect, atol=1e-9):
            raise RuntimeError(f'EXP-004 control not reproduced: found={found}, expected={expect}')
        say('EXP-004 pixel control EXACTLY reproduced')
    best = max(summary_rows, key=lambda r: (r['handmade_macro_f1'], r['official_accuracy']))
    result = {
        'experiment': 'EXP-005 SVC Normal HOG safe pilot',
        'train_size': len(ytr), 'validation_size': len(yval), 'official_test_used': False,
        'model_source': source, 'model_parameters': svc.get_params(),
        'official_dataset_sha256': sha,
        'results': records, 'selected_development_candidate': best['model'],
        'note': 'Subset pilot; repeated-use Hand-made development set is not an independent final test',
    }
    (output_dir / 'metrics.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    with (output_dir / 'comparison.csv').open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0]))
        writer.writeheader()
        writer.writerows(summary_rows)
    say('\nEXP-005 SAFE PILOT SUMMARY')
    for r in summary_rows:
        say(f'  {r["model"]:<27} official={r["official_accuracy"]:.6f} '
            f'handmade={r["handmade_accuracy"]:.6f} F1={r["handmade_macro_f1"]:.6f}')
    say('SELECTED DEVELOPMENT CANDIDATE:', best['model'])
    say('Saved:', output_dir)
    say('Official Test was NOT loaded.')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train-size', type=int, default=8000, choices=(2000, 4000, 8000, 16000))
    parser.add_argument('--val-size', type=int, default=3200, choices=(800, 1600, 3200, 6400))
    parser.add_argument('--batch-size', type=int, default=256, choices=(64, 128, 256, 512))
    run(parser.parse_args())


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")